# V1 Approach: HTML → Markdown → Chunk

> **This is the first prototype pipeline** built before the structured parser (`src/ingestion/parser/parser.py`).
>
> It converts a raw SEC iXBRL HTML filing directly to Markdown, detects section
> boundaries with a `## Item N.` regex, assigns each section a chunking strategy,
> and chunks prose with `RecursiveCharacterTextSplitter` and tables by pipe-line detection.
>
> **All data generated by this notebook is stored inside `old_approach/data/`.**
> Run cells top-to-bottom; each step reads from and writes to that local folder.
>
> ### Why this approach was replaced
> See `study/01.1_parser_appraoch.md` for the full story. Short version:
> - iXBRL tables produce noisy Markdown (spacer `| | |` cells, `$` split from number)
> - No way to verify whether a parsed number is correct
> - Section detection depends on regex on flat text — breaks for non-standard filings
> - No XBRL fact extraction — units/scale/sign/period are discarded
>
> ### Pipeline Steps
> 1. Download 10-K HTML from SEC EDGAR → `data/raw/`
> 2. Clean iXBRL HTML → flat Markdown → `data/processed_md_files/`
> 3. Detect sections, assign strategy, chunk → `data/chunks/`
> 4. Embed with OpenAI → store in Qdrant → `data/qdrant_storage/`
> 5. Retrieve top-k chunks for a query
> 6. Generate an answer with GPT


## Setup — path resolution

In [ ]:
from pathlib import Path

# All data lives inside old_approach/data/ so the notebook is self-contained.
NOTEBOOK_DIR = Path().resolve()   # cwd when notebook is run from its own folder
DATA_DIR     = NOTEBOOK_DIR / "data"

RAW_DIR    = DATA_DIR / "raw"
MD_DIR     = DATA_DIR / "processed_md_files"
CHUNKS_DIR = DATA_DIR / "chunks"
QDRANT_DIR = DATA_DIR / "qdrant_storage"

for d in [RAW_DIR, MD_DIR, CHUNKS_DIR, QDRANT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Data directories:")
for d in [RAW_DIR, MD_DIR, CHUNKS_DIR, QDRANT_DIR]:
    print(f"  {d}")


## Step 1 — Download 10-K HTML from SEC EDGAR

In [ ]:
import re

import requests


def download_latest_10k(CIK: str, output_dir: Path) -> Path:
    """Download the latest 10-K filing for a given CIK into output_dir."""
    HEADERS = {"User-Agent": "Satyajit Khot satyajit.khot@example.com"}

    resp = requests.get(
        f"https://data.sec.gov/submissions/CIK{CIK}.json",
        headers=HEADERS, timeout=30
    )
    resp.raise_for_status()
    data = resp.json()

    company_name = re.sub(r"[^\w]+", "_", data["name"].strip())
    recent = data["filings"]["recent"]

    for i, form in enumerate(recent["form"]):
        if form == "10-K":
            accession   = recent["accessionNumber"][i]
            primary_doc = recent["primaryDocument"][i]
            filing_date = recent["filingDate"][i]
            break
    else:
        raise RuntimeError("No 10-K filing found.")

    print(f"Filing date : {filing_date}")
    print(f"Accession   : {accession}")
    print(f"Document    : {primary_doc}")

    cik_no_zeros        = str(int(CIK))
    accession_no_dashes = accession.replace("-", "")
    document_url = (
        f"https://www.sec.gov/Archives/edgar/data/"
        f"{cik_no_zeros}/{accession_no_dashes}/{primary_doc}"
    )
    print(f"\nDownloading: {document_url}")

    report = requests.get(document_url, headers=HEADERS, timeout=60)
    report.raise_for_status()

    output_dir.mkdir(parents=True, exist_ok=True)
    out_file = output_dir / f"{company_name}_10K_{filing_date}.html"
    out_file.write_bytes(report.content)
    print(f"\nSaved: {out_file.resolve()}")
    return out_file


html_file = download_latest_10k(CIK="0000320193", output_dir=RAW_DIR)


## Step 2 — Clean iXBRL HTML → Markdown

The 10-K is an **Inline XBRL (iXBRL)** file. Every number appears twice: once as
visible HTML and once wrapped in machine-readable XBRL tags. We strip the XBRL layer,
remove noise, promote `Item N.` headings to `<h2>` tags, then convert to Markdown.

**Key functions:**
- `clean_ixbrl_html()` — removes hidden blocks, XBRL tags, boilerplate text
- `mark_item_headings()` — converts "Item 1. Business" text nodes to `<h2>`
- `html_to_markdown()` — uses `markdownify` to convert cleaned HTML → Markdown
- `normalize_markdown()` — collapses whitespace, removes boilerplate patterns
- `process_filing()` — wires all steps together, writes `.md` to `data/processed_md_files/`

> **Limitation discovered:** `markdownify` mirrors the iXBRL HTML layout exactly,
> so tables arrive with spacer `| | |` cells and `$` signs as separate cells.
> `clean_table_text()` in Step 3 partially fixes this but can't fully recover the
> structural information lost during HTML→Markdown flattening.


In [ ]:
import warnings
from collections import Counter
from pathlib import Path

from bs4 import BeautifulSoup, Comment, XMLParsedAsHTMLWarning
from markdownify import MarkdownConverter

warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)


def extract_company_name(html_path: Path) -> str:
    stem = html_path.stem
    m = re.match(r"^(.+?)_10K_\d{4}-\d{2}-\d{2}$", stem)
    return m.group(1).replace("_", " ").strip() if m else None


def clean_ixbrl_html(html_path: Path, company_name: str = None) -> BeautifulSoup:
    """Parse an SEC iXBRL filing and strip XBRL/boilerplate noise.

    Uses lxml-xml (XML mode) because SEC filings use self-closing tags on
    non-void elements (e.g. <td />) — only valid in XML, not HTML.
    """
    with open(html_path, "r", encoding="utf-8") as f:
        soup = BeautifulSoup(f, "lxml-xml")

    # 1. Remove hidden XBRL metadata blocks (display:none, <ix:header>)
    for hidden in soup.find_all(
        style=lambda s: s and "display:none" in s.replace(" ", "").lower()
    ):
        hidden.decompose()
    for tag in soup.find_all(["ix:header"]):
        tag.decompose()

    # 2. Remove XML/HTML comments
    for c in soup.find_all(string=lambda s: isinstance(s, Comment)):
        c.extract()

    # 3. Remove non-content tags
    for tag in soup(["script", "style", "meta", "link", "noscript", "svg"]):
        tag.decompose()

    # 4. Unwrap inline XBRL tags — keep text, drop the wrapper
    for tag in soup.find_all(lambda t: t.name and t.name.lower().startswith("ix:")):
        tag.unwrap()

    # 5. Unwrap internal anchor links (href="#...") — keep visible text
    for a in soup.find_all("a", href=True):
        if a["href"].startswith("#"):
            a.unwrap()

    # 6. Strip leftover XBRL-specific attributes
    xbrl_attrs = ["contextref", "unitref", "decimals", "scale", "format", "sign", "name"]
    for tag in soup.find_all(True):
        for attr in list(tag.attrs):
            if attr.lower() in xbrl_attrs:
                del tag.attrs[attr]

    # 7. Remove known boilerplate text nodes
    patterns = [
        r"^\s*Table of Contents\s*$",
        r"^\s*Page\s*\d+\s*$",
        r"^\s*.{1,80}\s*\|\s*\d{4}\s*Form\s*10-K\s*\|\s*(Page\s*)?\d+\s*$",
    ]
    if company_name:
        patterns.append(
            rf"^\s*{re.escape(company_name)}\s*\|\s*\d{{4}}\s*Form\s*10-K\s*\|\s*Page\s*\d+\s*$"
        )
    for node in soup.find_all(string=True):
        if any(re.match(p, node.strip(), re.IGNORECASE) for p in patterns):
            node.extract()

    return soup


def mark_item_headings(soup: BeautifulSoup) -> BeautifulSoup:
    """Convert 'Item N. Title' text nodes into <h2> tags."""
    pattern = re.compile(r"^Item\s+(\d+[A-C]?)\.\s*[\xa0\s]*(.*)$")
    for node in list(soup.find_all(string=True)):
        text = node.strip()
        if not text:
            continue
        m = pattern.match(text)
        if not m or node.find_parent("table") is not None:
            continue
        h2 = soup.new_tag("h2")
        h2.string = f"Item {m.group(1)}. {m.group(2).strip()}"
        node.replace_with(h2)
    return soup


def html_to_markdown(soup: BeautifulSoup) -> str:
    converter = MarkdownConverter(heading_style="ATX", strip=["script", "style"])
    md = converter.convert_soup(soup)
    return re.sub(r"\n{3,}", "\n\n", md).strip()


def normalize_markdown(text: str, company_name: str = None) -> str:
    text = text.replace("\xa0", " ")
    lines = text.split("\n")
    merged, buf = [], []

    def flush():
        if buf:
            merged.append(" ".join(buf))
            buf.clear()

    for line in lines:
        stripped = line.strip()
        if stripped.startswith("|") or stripped == "":
            flush()
            merged.append(line)
        else:
            buf.append(stripped)
    flush()

    text = "\n".join(merged)
    text = re.sub(r" {2,}", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r"\n---\n", "\n", text)
    if company_name:
        text = re.sub(
            rf"{re.escape(company_name)}\s*\|\s*\d{{4}}\s*Form\s*10-K\s*\|\s*\d+",
            "", text
        )
    return text.strip()


def is_junk_table_row(line: str) -> bool:
    stripped = line.strip()
    if not stripped.startswith("|"):
        return False
    cells = [c.strip() for c in stripped.strip("|").split("|")]
    return all(c == "" or re.fullmatch(r"-{2,}", c) for c in cells)


def strip_empty_table_rows(md: str) -> str:
    return "\n".join(line for line in md.split("\n") if not is_junk_table_row(line))


def process_filing(html_path: Path, output_dir: Path) -> Path:
    """Full pipeline: iXBRL HTML → cleaned Markdown file saved to output_dir."""
    output_dir.mkdir(parents=True, exist_ok=True)
    company_name = extract_company_name(html_path)

    soup    = clean_ixbrl_html(html_path, company_name=company_name)
    soup    = mark_item_headings(soup)
    md_text = html_to_markdown(soup)
    md_text = normalize_markdown(md_text, company_name=company_name)
    md_text = strip_empty_table_rows(md_text)

    output_path = output_dir / f"{html_path.stem}.md"
    output_path.write_text(md_text, encoding="utf-8")
    print(f"Markdown saved to: {output_path.resolve()}")
    return output_path


md_file = process_filing(html_file, MD_DIR)
print(f"\nSize: {md_file.stat().st_size / 1024:.1f} KB")
print("\n--- First 3,000 characters ---\n")
print(md_file.read_text(encoding="utf-8")[:3000])

## Step 3 — Detect Sections & Chunk

### 3a — Load and inspect the Markdown

In [ ]:
output_file = md_file
text = output_file.read_text(encoding="utf-8")
print(f"Total characters : {len(text):,}")
print(f"Total lines      : {text.count(chr(10)):,}")
print(f"Approx words     : {len(text.split()):,}")
print("\n--- First 500 characters ---")
print(text[:500])

### 3b — Find `## Item N.` headings

In [ ]:
pattern = re.compile(r"^##\s+Item\s+(\d+[A-C]?)\.\s+(.+)$", re.MULTILINE)
matches = list(pattern.finditer(text))

print(f"Found {len(matches)} Item sections\n")
for m in matches:
    print(f"  Item {m.group(1):5s} | pos={m.start():6d} | {m.group(2).strip()[:60]}")


### 3c — Build section map

In [ ]:
sections = {}
for i, m in enumerate(matches):
    item_key   = f"Item {m.group(1)}"
    section_end = matches[i + 1].start() if i + 1 < len(matches) else len(text)
    content    = text[m.start():section_end]
    sections[item_key] = {
        "name"   : m.group(2).strip(),
        "content": content,
        "length" : len(content),
    }

print(f"{'Section':<12} {'Name':<55} {'Chars':>8}")
print("-" * 80)
for key, val in sections.items():
    print(f"{key:<12} {val['name'][:55]:<55} {val['length']:>8,}")


### 3d — Assign chunking strategy

| Strategy | Condition | Method |
|---|---|---|
| `skip` | < 200 chars, or exhibit index / signatures | Ignored |
| `recursive` | Pure prose | `RecursiveCharacterTextSplitter` |
| `table_aware` | Contains `|`-prefixed lines | Tables atomic, prose split recursively |


In [ ]:
def has_tables(content: str) -> bool:
    return any(line.strip().startswith("|") for line in content.split("\n"))


SECTION_STRATEGY = {}
for key, val in sections.items():
    chars = val["length"]
    has_t = has_tables(val["content"])
    if chars < 200:
        strategy = "skip"
    elif key in ("Item 15", "Item 16"):
        strategy = "skip"   # exhibit index / signature block — not useful for RAG
    elif has_t:
        strategy = "table_aware"
    else:
        strategy = "recursive"
    SECTION_STRATEGY[key] = strategy
    print(f"{key:<12} -> {strategy:<15}  (chars={chars:,}, tables={has_t})")

counts = Counter(SECTION_STRATEGY.values())
print()
for s, n in counts.items():
    print(f"  {s:<15} -> {n} sections")

### 3e — Chunking functions

> **Problem visible here:** `clean_table_text()` is a post-hoc fix for noisy Markdown
> tables. The root cause — iXBRL spacer `<td>` cells becoming `| | |` in Markdown —
> was only properly solved by parsing the HTML DOM directly in the structured parser.


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", ". ", " ", ""]
)


def clean_table_text(table_text: str) -> str:
    """Strip empty spacer cells and merge lone '$' signs with their number.

    NOTE: This is a post-hoc cleanup. The root cause is that iXBRL tables use
    extra <td> elements as visual spacers; after HTML->Markdown conversion they
    become empty '| |' cells. The structured parser (src/ingestion/parser/parser.py)
    solves this properly by parsing the DOM before serialisation.
    """
    lines   = table_text.split("\n")
    cleaned = []
    for line in lines:
        cells = [c.strip() for c in line.split("|")]
        cells = [c for c in cells if c and not re.fullmatch(r"-{2,}", c)]
        merged, prefix = [], ""
        for c in cells:
            if c == "$":
                prefix = "$"
                continue
            if c == "%" and merged:
                merged[-1] += "%"
                continue
            merged.append(prefix + c)
            prefix = ""
        if prefix:
            merged.append(prefix)
        if not merged:
            continue
        cleaned.append("| " + " | ".join(merged) + " |")
    return "\n".join(cleaned)


def split_into_blocks(content: str) -> list:
    """Split a section into alternating prose and table blocks."""
    blocks, current_type, current_lines = [], None, []
    for line in content.split("\n"):
        line_type = "table" if line.strip().startswith("|") else "prose"
        if line_type != current_type:
            if current_lines:
                block_text = "\n".join(current_lines).strip()
                if block_text:
                    blocks.append({"type": current_type, "text": block_text})
            current_lines, current_type = [line], line_type
        else:
            current_lines.append(line)
    if current_lines:
        block_text = "\n".join(current_lines).strip()
        if block_text:
            blocks.append({"type": current_type, "text": block_text})
    return blocks


def chunk_prose_section(section_key: str, section_data: dict, splitter) -> list:
    """Chunk a prose-only section with RecursiveCharacterTextSplitter."""
    raw_chunks = splitter.split_text(section_data["content"])
    return [
        {
            "text": chunk,
            "metadata": {
                "source"      : output_file.name,
                "company"     : "Apple",
                "year"        : 2025,
                "section"     : section_key,
                "section_name": section_data["name"],
                "content_type": "prose",
                "chunk_index" : i,
                "chunk_total" : len(raw_chunks),
            },
        }
        for i, chunk in enumerate(raw_chunks)
    ]


def chunk_table_aware_section(section_key: str, section_data: dict, splitter) -> list:
    """Chunk a mixed prose+table section.
    - Tables: kept as ONE chunk (never split)
    - Prose: split with the recursive splitter
    """
    blocks = split_into_blocks(section_data["content"])
    chunks = []
    for block in blocks:
        if block["type"] == "table":
            chunks.append({
                "text": clean_table_text(block["text"]),
                "metadata": {
                    "source"      : output_file.name,
                    "company"     : "Apple",
                    "year"        : 2025,
                    "section"     : section_key,
                    "section_name": section_data["name"],
                    "content_type": "table",
                    "chunk_index" : len(chunks),
                    "chunk_total" : None,
                },
            })
        else:
            for prose_chunk in splitter.split_text(block["text"]):
                chunks.append({
                    "text": prose_chunk,
                    "metadata": {
                        "source"      : output_file.name,
                        "company"     : "Apple",
                        "year"        : 2025,
                        "section"     : section_key,
                        "section_name": section_data["name"],
                        "content_type": "prose",
                        "chunk_index" : len(chunks),
                        "chunk_total" : None,
                    },
                })
    total = len(chunks)
    for c in chunks:
        c["metadata"]["chunk_total"] = total
    return chunks


def chunk_document(sections: dict, section_strategy: dict, splitter) -> list:
    """Run all sections through their assigned strategy, return flat chunk list."""
    all_chunks = []
    for key, strategy in section_strategy.items():
        if strategy == "skip":
            continue
        elif strategy == "recursive":
            new = chunk_prose_section(key, sections[key], splitter)
        else:
            new = chunk_table_aware_section(key, sections[key], splitter)
        all_chunks.extend(new)
        print(f"  {key:<12} [{strategy:<12}] -> {len(new):>3} chunks")
    return all_chunks


print("Chunking Apple 10-K...\n")
all_chunks = chunk_document(sections, SECTION_STRATEGY, splitter)

prose_count = sum(1 for c in all_chunks if c["metadata"]["content_type"] == "prose")
table_count = sum(1 for c in all_chunks if c["metadata"]["content_type"] == "table")
print(f"\n{'='*45}")
print(f"  Total chunks : {len(all_chunks)}")
print(f"  Prose chunks : {prose_count}")
print(f"  Table chunks : {table_count}")


### 3f — Save chunks to JSON

In [ ]:
import json

chunks_file = CHUNKS_DIR / "apple_10K.json"

output = {
    "metadata": {
        "source"       : output_file.name,
        "company"      : "Apple",
        "year"         : 2025,
        "total_chunks" : len(all_chunks),
        "prose_chunks" : prose_count,
        "table_chunks" : table_count,
        "chunk_size"   : 1000,
        "chunk_overlap": 200,
    },
    "chunks": all_chunks,
}

chunks_file.write_text(json.dumps(output, indent=2), encoding="utf-8")
print(f"Saved to : {chunks_file.resolve()}")
print(f"File size: {chunks_file.stat().st_size / 1024:.1f} KB")


## Step 4 — Embed Chunks & Store in Qdrant

Uses `text-embedding-3-small` via the OpenAI API.
Set `OPENAI_API_KEY` in your environment before running this section.


In [ ]:
import time

import numpy as np
from openai import OpenAI

# Load chunks
with open(chunks_file, "r", encoding="utf-8") as f:
    data = json.load(f)

print(f"Chunks loaded: {len(data['chunks'])}")

# OpenAI client — reads OPENAI_API_KEY from environment
client_openai   = OpenAI()
EMBEDDING_MODEL = "text-embedding-3-small"   # 1536-dim

# Quick connectivity test
test = client_openai.embeddings.create(model=EMBEDDING_MODEL, input=["test"])
VECTOR_DIM = len(test.data[0].embedding)
print(f"API connected! Embedding dim: {VECTOR_DIM}")


In [ ]:
# Embed all chunks in batches of 32
texts      = [c["text"] for c in data["chunks"]]
BATCH_SIZE = 32
all_embeddings = []
start = time.time()

for i in range(0, len(texts), BATCH_SIZE):
    batch = texts[i : i + BATCH_SIZE]
    resp  = client_openai.embeddings.create(model=EMBEDDING_MODEL, input=batch)
    vecs  = [item.embedding for item in sorted(resp.data, key=lambda x: x.index)]
    all_embeddings.extend(vecs)
    print(f"  Embedded {min(i + BATCH_SIZE, len(texts))}/{len(texts)} chunks...")

embeddings = np.array(all_embeddings)
print(f"\nDone! {time.time()-start:.1f}s  |  Shape: {embeddings.shape}")


In [ ]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, PointStruct, VectorParams

COLLECTION = "apple_10k_v1"

qdrant = QdrantClient(path=str(QDRANT_DIR))

# Recreate collection (idempotent)
if qdrant.collection_exists(COLLECTION):
    qdrant.delete_collection(COLLECTION)

qdrant.create_collection(
    collection_name=COLLECTION,
    vectors_config=VectorParams(size=VECTOR_DIM, distance=Distance.COSINE)
)

points = [
    PointStruct(
        id=i,
        vector=embeddings[i].tolist(),
        payload={"text": data["chunks"][i]["text"], **data["chunks"][i]["metadata"]}
    )
    for i in range(len(data["chunks"]))
]

qdrant.upsert(collection_name=COLLECTION, points=points)
info = qdrant.get_collection(COLLECTION)
print(f"Uploaded {info.points_count} points to Qdrant")
print(f"Storage : {QDRANT_DIR}")


## Step 5 — Retrieve Chunks

In [ ]:
def retrieve(query: str, top_k: int = 5) -> list:
    """Embed a query and retrieve top-k matching chunks from Qdrant."""
    q_vec = client_openai.embeddings.create(
        model=EMBEDDING_MODEL, input=[query]
    ).data[0].embedding
    return qdrant.query_points(
        collection_name=COLLECTION, query=q_vec, limit=top_k
    ).points


query = "What was Apple's total revenue in 2025?"
hits  = retrieve(query)

print(f"Query: '{query}'\n")
for i, hit in enumerate(hits):
    print(f"Rank {i+1} | score={hit.score:.4f} | {hit.payload['section']} ({hit.payload['content_type']})")
    print(f"  {hit.payload['text'][:200]}...")
    print()


## Step 6 — Generate an Answer

In [ ]:
def build_context(hits: list) -> str:
    parts = []
    for hit in hits:
        section = hit.payload["section"]
        ctype   = hit.payload["content_type"]
        parts.append(f"[{section} | {ctype}]\n{hit.payload['text']}")
    return "\n\n---\n\n".join(parts)


def answer(query: str, top_k: int = 5) -> str:
    """Retrieve relevant chunks and generate an answer with GPT."""
    hits    = retrieve(query, top_k=top_k)
    context = build_context(hits)

    system_prompt = (
        "You are a financial analyst assistant. Answer the user's question using ONLY "
        "the context provided below. If the answer is not in the context, say so clearly."
    )
    user_prompt = f"Context:\n{context}\n\nQuestion: {query}"

    response = client_openai.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt},
        ],
        temperature=0.1,
    )
    return response.choices[0].message.content


# Try a few questions
questions = [
    "What was Apple's total revenue in 2025?",
    "What were Apple's main risk factors?",
    "How much did Apple spend on R&D in 2025?",
]

for q in questions:
    print(f"Q: {q}")
    print(f"A: {answer(q)}")
    print("-" * 60)
